# W3D5 — Benchmark Harness + Cost & Scale-Out
### Kaggle notebook — Fay Alaamri

This notebook combines both Day 5 labs:

1. **Main Lab — The Benchmark Harness: Find the Knee**
2. **Extra Lab — Cost per Million Tokens and Scale-Out**

### Kaggle setup
Open **Settings → Accelerator → GPU**, then run the cells in order.

The benchmark keeps the model/engine/launch flags fixed and changes only **concurrency**.

In [1]:
# 1 — Check GPU
!nvidia-smi

Thu Sep  3 09:49:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2 — Lab configuration

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

# Prediction card: set these BEFORE measuring.
PREDICTED_KNEE = 4
TARGET_P95_S = 3.0

# Representative value from the extra lab.
# Replace with a real GPU rate if your instructor gives you one.
GPU_HOURLY_USD = 0.35

CONCURRENCY_LEVELS = [1, 2, 4, 8, 16]
REQUESTS_PER_LEVEL = 20

print("Model:", MODEL_ID)
print("Predicted knee:", PREDICTED_KNEE)
print("p95 SLO:", TARGET_P95_S, "seconds")

Model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
Predicted knee: 4
p95 SLO: 3.0 seconds


## Install and launch vLLM

The course says to use the **exact model and launch flags locked in W3D4**. The uploaded guide mentions `Qwen/Qwen2.5-1.5B-Instruct-AWQ` as a possible locked model, so it is the default here.

If your actual `model-lock.md` used different flags, edit the configuration before benchmarking.

In [3]:
# 3 — Install
!pip -q install -U vllm httpx requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 93.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 102.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 46.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━

In [5]:
# 4 — Imports
import sys, os, json, math, time, subprocess, asyncio
from time import perf_counter
import requests, httpx
import vllm

print("Python:", sys.version.split()[0])
print("vLLM:", vllm.__version__)
print("Environment ready")

Python: 3.12.13
vLLM: 0.28.0
Environment ready


## Fixed prompts

The official course harness uses 20 fixed prompts so every concurrency level receives the same workload. The actual course `prompts.txt` was not included in the uploaded lab files, so this notebook creates 20 fixed prompts.

If you have the official `prompts.txt`, upload it and use that instead.

In [6]:
# 5 — Create prompts.txt
prompts = [
    "Explain GPU memory bandwidth in simple terms.",
    "What is the difference between prefill and decode in LLM inference?",
    "Explain KV cache in three short sentences.",
    "Why can decode be memory-bandwidth bound?",
    "What does TTFT measure?",
    "What does TPOT measure?",
    "Explain concurrency in an inference server.",
    "What is p95 latency and why is it useful?",
    "What does tokens per second measure?",
    "Explain the difference between throughput and latency.",
    "Why can increasing concurrency improve throughput?",
    "What happens when an inference server becomes saturated?",
    "Explain arithmetic intensity in simple terms.",
    "What is the roofline model used for?",
    "Why can low GPU utilization be misleading?",
    "What is quantization and why can it help inference?",
    "Explain static batching versus continuous batching.",
    "What is a benchmark knee?",
    "Why should capacity respect an SLO?",
    "Explain why peak throughput may not be safe capacity.",
]

assert len(prompts) == 20
with open("prompts.txt", "w") as f:
    for p in prompts:
        f.write(p + "\n")

print("prompts.txt created:", len(prompts), "prompts")

prompts.txt created: 20 prompts


In [7]:
# 6 — Launch the locked model
!pkill -f "vllm.entrypoints.openai.api_server" || true

SERVER_ARGS = [
    "--model", MODEL_ID,
    "--host", "0.0.0.0",
    "--port", "8000",
    "--dtype", "half",
    "--quantization", "awq",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "4096",
]

cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    *SERVER_ARGS,
]

log_file = open("vllm_server.log", "w")
server_process = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Server PID:", server_process.pid)
print(" ".join(cmd))

Server PID: 205
/usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --host 0.0.0.0 --port 8000 --dtype half --quantization awq --gpu-memory-utilization 0.90 --max-model-len 4096


In [8]:
# 7 — Health poll
BASE_URL = "http://127.0.0.1:8000"
deadline = time.time() + 600

while time.time() < deadline:
    if server_process.poll() is not None:
        print("Server exited. Log:")
        !tail -100 vllm_server.log
        raise RuntimeError("vLLM failed to start")

    try:
        r = requests.get(BASE_URL + "/v1/models", timeout=5)
        if r.status_code == 200:
            print("HEALTHY — server ready")
            print(r.json())
            break
    except Exception:
        pass

    print("Waiting...")
    time.sleep(5)
else:
    !tail -100 vllm_server.log
    raise TimeoutError("Server did not become healthy")

Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
Waiting...
HEALTHY — server ready
{'object': 'list', 'data': [{'id': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'object': 'model', 'created': 1788430072, 'owned_by': 'vllm', 'root': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'parent': None, 'max_model_len': 4096, 'permission': [{'id': 'modelperm-91a99f1c2f2d0b71', 'object': 'model_permission', 'created': 1788430072, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


# LAB 1 — Benchmark Harness: Find the Knee

We sweep **1 → 2 → 4 → 8 → 16** and measure:

- `tokens_per_s`
- `ttft_p50_s`
- `ttft_p95_s`
- `latency_p95_s`
- `errors`

The knee is the **highest tested concurrency whose p95 latency still meets the SLO**.

In [9]:
# 8 — Benchmark harness

def percentile(values, p):
    values = sorted(values)
    if not values:
        return 0.0
    if len(values) == 1:
        return float(values[0])
    k = (len(values) - 1) * p
    lo, hi = math.floor(k), math.ceil(k)
    if lo == hi:
        return float(values[lo])
    return float(values[lo] * (hi-k) + values[hi] * (k-lo))


async def one_request(client, prompt):
    start = perf_counter()
    first_token = None
    completion_tokens = 0

    payload = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 64,
        "temperature": 0.0,
        "stream": True,
        "stream_options": {"include_usage": True},
    }

    try:
        async with client.stream(
            "POST",
            BASE_URL + "/v1/chat/completions",
            json=payload,
            timeout=120.0,
        ) as response:
            response.raise_for_status()

            async for line in response.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[6:].strip()
                if data == "[DONE]":
                    break

                obj = json.loads(data)
                choices = obj.get("choices", [])
                if choices:
                    content = choices[0].get("delta", {}).get("content")
                    if content and first_token is None:
                        first_token = perf_counter()

                usage = obj.get("usage")
                if usage:
                    completion_tokens = int(
                        usage.get("completion_tokens", completion_tokens)
                    )

        end = perf_counter()
        if first_token is None:
            first_token = end

        return {
            "ok": True,
            "ttft_s": first_token - start,
            "latency_s": end - start,
            "completion_tokens": completion_tokens,
        }

    except Exception as e:
        return {
            "ok": False,
            "ttft_s": None,
            "latency_s": perf_counter() - start,
            "completion_tokens": 0,
            "error": repr(e),
        }


async def warm_up():
    async with httpx.AsyncClient() as client:
        r = await one_request(client, prompts[0])
    print("Warm-up:", "OK" if r["ok"] else r.get("error"))


async def benchmark_level(concurrency, requests_per_level):
    semaphore = asyncio.Semaphore(concurrency)

    async with httpx.AsyncClient(
        limits=httpx.Limits(
            max_connections=max(100, concurrency * 4),
            max_keepalive_connections=max(20, concurrency * 2),
        )
    ) as client:

        async def bounded(i):
            async with semaphore:
                return await one_request(
                    client,
                    prompts[i % len(prompts)],
                )

        start = perf_counter()
        results = await asyncio.gather(
            *[bounded(i) for i in range(requests_per_level)]
        )
        elapsed = perf_counter() - start

    good = [r for r in results if r["ok"]]
    ttfts = [r["ttft_s"] for r in good if r["ttft_s"] is not None]
    latencies = [r["latency_s"] for r in good]
    tokens = sum(r["completion_tokens"] for r in good)

    return {
        "concurrency": int(concurrency),
        "tokens_per_s": tokens / elapsed if elapsed else 0.0,
        "ttft_p50_s": percentile(ttfts, 0.50),
        "ttft_p95_s": percentile(ttfts, 0.95),
        "latency_p95_s": percentile(latencies, 0.95),
        "errors": int(len(results) - len(good)),
        "requests": int(requests_per_level),
        "elapsed_s": elapsed,
        "completion_tokens": int(tokens),
    }

print("Harness ready")

Harness ready


In [10]:
# 9 — Initial warm-up
await warm_up()

Warm-up: OK


In [11]:
# 10 — Run the sweep
levels = []

for c in CONCURRENCY_LEVELS:
    print("\n===== concurrency", c, "=====")
    await warm_up()

    L = await benchmark_level(c, REQUESTS_PER_LEVEL)
    levels.append(L)

    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )


===== concurrency 1 =====
Warm-up: OK
c= 1  tok/s=  130.7  ttft_p95=0.027  lat_p95=0.492  errors=0

===== concurrency 2 =====
Warm-up: OK
c= 2  tok/s=  262.8  ttft_p95=0.046  lat_p95=0.494  errors=0

===== concurrency 4 =====
Warm-up: OK
c= 4  tok/s=  502.9  ttft_p95=0.051  lat_p95=0.515  errors=0

===== concurrency 8 =====
Warm-up: OK
c= 8  tok/s=  795.9  ttft_p95=0.057  lat_p95=0.556  errors=0

===== concurrency 16 =====
Warm-up: OK
c=16  tok/s= 1057.2  ttft_p95=0.086  lat_p95=0.688  errors=0


In [12]:
# 11 — Save bench_report.json
bench_report = {
    "model": MODEL_ID,
    "runs": [{
        "levels": levels
    }]
}

with open("bench_report.json", "w") as f:
    json.dump(bench_report, f, indent=2)

print("bench_report.json written")

bench_report.json written


In [13]:
# 12 — Read report
levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )

c= 1  tok/s=  130.7  ttft_p95=0.027  lat_p95=0.492  errors=0
c= 2  tok/s=  262.8  ttft_p95=0.046  lat_p95=0.494  errors=0
c= 4  tok/s=  502.9  ttft_p95=0.051  lat_p95=0.515  errors=0
c= 8  tok/s=  795.9  ttft_p95=0.057  lat_p95=0.556  errors=0
c=16  tok/s= 1057.2  ttft_p95=0.086  lat_p95=0.688  errors=0


In [15]:
# 13 — Find knee
under = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under,
    key=lambda L: L["concurrency"],
) if under else None

print("TARGET P95 =", TARGET_P95_S)
print("KNEE =", knee)

if knee and knee["concurrency"] == max(CONCURRENCY_LEVELS):
    print("Sweep-bounded: the real knee may be >= 16.")

TARGET P95 = 3.0
KNEE = {'concurrency': 16, 'tokens_per_s': 1057.1539793848872, 'ttft_p50_s': 0.07607493799991971, 'ttft_p95_s': 0.08633749399994031, 'latency_p95_s': 0.6875651218998996, 'errors': 0, 'requests': 20, 'elapsed_s': 1.2041765200001464, 'completion_tokens': 1273}
Sweep-bounded: the real knee may be >= 16.


In [16]:
# 14 — Save knee.json
with open("knee.json", "w") as f:
    json.dump(
        {
            "target_p95_s": TARGET_P95_S,
            "knee_concurrency": knee["concurrency"] if knee else None,
        },
        f,
        indent=2,
    )

print(open("knee.json").read())

{
  "target_p95_s": 3.0,
  "knee_concurrency": 16
}


In [17]:
# 15 — Create capacity-note.md

if knee:
    rps = knee["requests"] / knee["elapsed_s"]

    note = f"""# W3D5 Capacity Note

- Locked model: {MODEL_ID}
- Target p95 SLO: {TARGET_P95_S:.3f} s
- Predicted knee: concurrency {PREDICTED_KNEE}
- Knee concurrency: {knee["concurrency"]}
- Tokens/s at knee: {knee["tokens_per_s"]:.2f}
- p95 at knee: {knee["latency_p95_s"]:.3f} s
- Max sustainable request rate: approximately {rps:.2f} requests/s
- Limiting family: inspect GPU utilisation, clocks, memory-bandwidth symptoms, and host overhead before choosing compute / memory / overhead.
- Why knee, not peak: the knee is the highest tested concurrency that still meets the p95 SLO, so it is the capacity we can honestly promise.
"""
else:
    note = f"""# W3D5 Capacity Note

- Locked model: {MODEL_ID}
- Target p95 SLO: {TARGET_P95_S:.3f} s
- Predicted knee: concurrency {PREDICTED_KNEE}
- Knee concurrency: none
- Tokens/s at knee: none
- Max sustainable request rate: none at this SLO
- Limiting family: requires diagnosis
- Why knee, not peak: no tested level meets this SLO, so peak throughput is not safe capacity.
"""

with open("capacity-note.md", "w") as f:
    f.write(note)

print(note)

# W3D5 Capacity Note

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 SLO: 3.000 s
- Predicted knee: concurrency 4
- Knee concurrency: 16
- Tokens/s at knee: 1057.15
- p95 at knee: 0.688 s
- Max sustainable request rate: approximately 16.61 requests/s
- Limiting family: inspect GPU utilisation, clocks, memory-bandwidth symptoms, and host overhead before choosing compute / memory / overhead.
- Why knee, not peak: the knee is the highest tested concurrency that still meets the p95 SLO, so it is the capacity we can honestly promise.



In [18]:
# 16 — Main Lab Green Check
problems = []

assert os.path.exists("bench_report.json")
assert os.path.exists("knee.json")
assert os.path.exists("capacity-note.md")

report = json.load(open("bench_report.json"))
lvls = report["runs"][-1]["levels"]

if len(lvls) < 4:
    problems.append("Need at least 4 levels")

required = {
    "concurrency", "tokens_per_s",
    "ttft_p50_s", "ttft_p95_s",
    "latency_p95_s", "errors"
}

for L in lvls:
    if not required.issubset(L):
        problems.append("Missing required benchmark fields")
    if not isinstance(L["errors"], int) or L["errors"] < 0:
        problems.append("errors must be non-negative integers")

k = json.load(open("knee.json"))
if k["target_p95_s"] <= 0:
    problems.append("target_p95_s must be > 0")
if k["knee_concurrency"] is None or k["knee_concurrency"] < 1:
    problems.append("No valid knee")

if "FILL:" in open("capacity-note.md").read():
    problems.append("FILL placeholder remains")

if not any(L["tokens_per_s"] > 0 for L in lvls):
    problems.append("No positive throughput")

if problems:
    print("GREEN CHECK: FAIL")
    for p in problems:
        print("-", p)
else:
    print("GREEN CHECK: PASS")

GREEN CHECK: PASS


# LAB 2 — Cost per Million Tokens and Scale-Out

We now use `bench_report.json`.

The lab formula is:

**tokens/hour = tokens/s × 3600**

**million tokens/hour = tokens/hour ÷ 1,000,000**

**cost per million tokens = GPU hourly cost ÷ million tokens/hour**

Then the safe knee becomes one scale-out capacity unit.

In [20]:
# 17 — Cost function
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

priced_levels = []

for L in levels:
    x = dict(L)
    x["cost_per_million_tokens_usd"] = (
        cost_per_million_tokens(
            L["tokens_per_s"],
            GPU_HOURLY_USD,
        )
        if L["tokens_per_s"] > 0
        else None
    )
    priced_levels.append(x)

for L in priced_levels:
    print(
        f"c={L['concurrency']:>2} | "
        f"{L['tokens_per_s']:.2f} tok/s | "
        f"p95={L['latency_p95_s']:.3f}s | "
        f"cost/1M=${L['cost_per_million_tokens_usd']}"
    )

c= 1 | 130.69 tok/s | p95=0.492s | cost/1M=$0.7439
c= 2 | 262.79 tok/s | p95=0.494s | cost/1M=$0.37
c= 4 | 502.92 tok/s | p95=0.515s | cost/1M=$0.1933
c= 8 | 795.91 tok/s | p95=0.556s | cost/1M=$0.1222
c=16 | 1057.15 tok/s | p95=0.688s | cost/1M=$0.092


In [21]:
# 18 — Safe knee for cost/scale-out
under_target = [
    L for L in priced_levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

cost_knee = max(
    under_target,
    key=lambda L: L["concurrency"],
) if under_target else None

if cost_knee is None:
    raise RuntimeError("No level meets the p95 SLO.")

print("Safe knee concurrency:", cost_knee["concurrency"])
print("Knee tokens/s:", round(cost_knee["tokens_per_s"], 2))
print("Knee p95:", round(cost_knee["latency_p95_s"], 3))
print("Knee $/1M tokens:", cost_knee["cost_per_million_tokens_usd"])

Safe knee concurrency: 16
Knee tokens/s: 1057.15
Knee p95: 0.688
Knee $/1M tokens: 0.092


In [22]:
# 19 — Scale-out plan
def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

K = cost_knee["tokens_per_s"]
multiples = [1.0, 1.5, 2.0, 3.0]
scale_out_plan = []

for multiple in multiples:
    required = multiple * K
    replicas = replicas_needed(required, K)

    scale_out_plan.append({
        "demand_multiple": multiple,
        "required_tokens_per_s": required,
        "replicas": replicas,
        "hourly_cost_usd": round(replicas * GPU_HOURLY_USD, 4),
        "effective_p95_s": cost_knee["latency_p95_s"],
    })

for row in scale_out_plan:
    print(
        f"{row['demand_multiple']:.1f}x | "
        f"required={row['required_tokens_per_s']:.2f} tok/s | "
        f"replicas={row['replicas']} | "
        f"hourly=${row['hourly_cost_usd']:.2f} | "
        f"effective p95≈{row['effective_p95_s']:.3f}s"
    )

1.0x | required=1057.15 tok/s | replicas=1 | hourly=$0.35 | effective p95≈0.688s
1.5x | required=1585.73 tok/s | replicas=2 | hourly=$0.70 | effective p95≈0.688s
2.0x | required=2114.31 tok/s | replicas=2 | hourly=$0.70 | effective p95≈0.688s
3.0x | required=3171.46 tok/s | replicas=3 | hourly=$1.05 | effective p95≈0.688s


In [23]:
# 20 — Save cost_report.json
cost_report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": priced_levels,
    "knee": cost_knee,
    "scale_out_plan": scale_out_plan,
}

with open("cost_report.json", "w") as f:
    json.dump(cost_report, f, indent=2)

print("cost_report.json written")
print(json.dumps(cost_report, indent=2))

cost_report.json written
{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 3.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 130.6857374765278,
      "ttft_p50_s": 0.025576559000000998,
      "ttft_p95_s": 0.026887455900032366,
      "latency_p95_s": 0.49246327704995563,
      "errors": 0,
      "requests": 20,
      "elapsed_s": 9.740925250000146,
      "completion_tokens": 1273,
      "cost_per_million_tokens_usd": 0.7439
    },
    {
      "concurrency": 2,
      "tokens_per_s": 262.7914164772865,
      "ttft_p50_s": 0.027364089499997135,
      "ttft_p95_s": 0.046042442849955026,
      "latency_p95_s": 0.49422849915008554,
      "errors": 0,
      "requests": 20,
      "elapsed_s": 4.84414604199992,
      "completion_tokens": 1273,
      "cost_per_million_tokens_usd": 0.37
    },
    {
      "concurrency": 4,
      "tokens_per_s": 502.9174086777372,
      "ttft_p50_s": 0.040490888000022096,
      "ttft_p95_s": 0.050754163550027445,
      "latency_p95_s": 0.515042635

In [24]:
# 21 — Extra Lab Green Check
problems = []
r = json.load(open("cost_report.json"))

for field in [
    "gpu_hourly_usd",
    "target_p95_s",
    "levels",
    "knee",
    "scale_out_plan",
]:
    if field not in r:
        problems.append("Missing " + field)

for L in r["levels"]:
    if L["tokens_per_s"] > 0:
        expected = cost_per_million_tokens(
            L["tokens_per_s"],
            r["gpu_hourly_usd"],
        )
        if L["cost_per_million_tokens_usd"] != expected:
            problems.append(
                "Cost calculation error at c="
                + str(L["concurrency"])
            )

K = r["knee"]["tokens_per_s"]

for row in r["scale_out_plan"]:
    expected_replicas = math.ceil(
        row["required_tokens_per_s"] / K
    )
    if row["replicas"] != expected_replicas:
        problems.append("Replica calculation error")

    expected_hourly = round(
        row["replicas"] * r["gpu_hourly_usd"],
        4,
    )
    if row["hourly_cost_usd"] != expected_hourly:
        problems.append("Hourly cost calculation error")

if problems:
    print("GREEN CHECK: FAIL")
    for p in problems:
        print("-", p)
else:
    print("GREEN CHECK: PASS")

GREEN CHECK: PASS


## Final mental model

**Benchmark → find safe knee → price the knee → demand grows → replicate safe units.**

Do **not** treat SLO-violating throughput beyond the knee as honest capacity.

In [25]:
# 22 — List final artifacts
for f in [
    "bench_report.json",
    "knee.json",
    "capacity-note.md",
    "cost_report.json",
]:
    print(f, "OK" if os.path.exists(f) else "MISSING")

bench_report.json OK
knee.json OK
capacity-note.md OK
cost_report.json OK


In [26]:
# 23 — Package the four outputs
import zipfile

with zipfile.ZipFile("W3D5_Fay_lab_artifacts.zip", "w") as z:
    for f in [
        "bench_report.json",
        "knee.json",
        "capacity-note.md",
        "cost_report.json",
    ]:
        if os.path.exists(f):
            z.write(f)

print("Created W3D5_Fay_lab_artifacts.zip")

Created W3D5_Fay_lab_artifacts.zip
